Forwardpass

In [1]:
import torch # importa a biblioteca PyTorch

Criação dos dados

In [5]:
N = 10 # nossa amostra de dados vai ter 10 elementos

# cada ponto tem um input de feature e um output de value
D_in = 1
D_out = 1

data = torch.randn(N, D_in) # cria uma amostra de dados aleatória com 10 elementos e 1 feature  
print(f"Dados de entrada: {data}")
true_W = torch.tensor([[2.]]) # cria o valor verdadeiro do peso
true_b = torch.tensor([1.]) # cria o valor verdadeiro do bias
y_true = data @ true_W + true_b + torch.randn(N, D_out) * 0.1 # cria o valor verdadeiro do output com um pouco de noise 

Dados de entrada: tensor([[-1.2452],
        [ 0.2071],
        [ 0.1153],
        [-1.2374],
        [ 0.5830],
        [-0.2096],
        [-1.6135],
        [-0.9269],
        [ 0.5755],
        [-1.0619]])


Inicialização dos parametros, o "cérebro" do modelo

In [7]:
W = torch.randn(D_in, D_out, requires_grad=True) # cria o peso inicial aleatório
b = torch.randn(D_out, requires_grad=True) # cria o bias inicial aleatório
print(f"Peso inicial: {W}")
print(f"peso verdadeiro: {true_W}")
print(f"Bias inicial: {b}")
print(f"bias verdadeiro: {true_b}")


Peso inicial: tensor([[1.3704]], requires_grad=True)
peso verdadeiro: tensor([[2.]])
Bias inicial: tensor([-0.2180], requires_grad=True)
bias verdadeiro: tensor([1.])


In [12]:
y_hat = data @ W + b # calcula o output estimado com os pesos e bias iniciais
print(f"Output estimado: {y_hat[:3]}") # pegando só as tres primeiras linhas pra facilitar a visualização
print(f"Output verdadeiro: {y_true[:3]}")

Output estimado: tensor([[-1.9245],
        [ 0.0658],
        [-0.0600]], grad_fn=<SliceBackward0>)
Output verdadeiro: tensor([[-1.2431],
        [ 1.4934],
        [ 1.3776]])


backward pass

In [ ]:
# calculando a lossa
error = y_hat - y_true # diferença entre o y predito e o y alvo
squared_error = error ** 2 # garante que o número seja postivo
loss = squared_error.mean() 
print(f"Loss: {loss}") # quanto mais próximo de zero, melhor

Loss: 1.1824371814727783


Calcula o gradiente para saber qual o proximo passo

IMPORTANTE: o número calculado indica a direção que o modelo tem que ir, exemplo: bias negativo -> aumento do bias -> diminuição da loss

In [14]:
loss.backward() # calcula o gradiente da loss em relação aos pesos e bias
print(f"Gradiente do peso: {W.grad}")
print(f"Gradiente do bias: {b.grad}")

Gradiente do peso: tensor([[0.1918]])
Gradiente do bias: tensor([-1.9448])


Agora que temos a direção, basta recalcular com base nesses novos dados

torch.no_grad() # não rastreia a atualização dos parametros. 
.grad.zero_() # reseta os gradientes a cada iteração

In [27]:
learning_rate = 0.01 # define o tamanho do passo
epochs = 100 # define o número de iterações

W, b = torch.randn(D_in, D_out, requires_grad=True), torch.randn(D_out, requires_grad=True) # reinicia os pesos e bias
#loop do treinamento
for epoch in range(epochs): 
    # Forward pass
    y_hat = data @ W + b
    loss = (y_hat - y_true).pow(2).mean() # calcula a loss
    loss.backward() # calcula o gradiente da loss em relação aos pesos e bias

    with torch.no_grad(): # desliga o autograd para não acumular os gradientes
        W -= learning_rate * W.grad # atualiza o peso
        b -= learning_rate * b.grad # atualiza o bias

        W.grad.zero_() # zera o gradiente do peso
        b.grad.zero_() # zera o gradiente do bias
    if epoch % 10 == 0: # imprime a cada 10 iterações
        print(f"Epoch {epoch}: Loss = {loss.item()}, Peso = {W.item()}, Bias = {b.item()}") # imprime a loss

print(f"Peso final: {W}")
print(f"Bias final: {b}")

Epoch 0: Loss = 0.5902682542800903, Peso = 1.0629016160964966, Bias = 0.3693504333496094
Epoch 10: Loss = 0.4949905276298523, Peso = 1.147658109664917, Bias = 0.4162615239620209
Epoch 20: Loss = 0.41604408621788025, Peso = 1.223272681236267, Bias = 0.4616684317588806
Epoch 30: Loss = 0.3503473401069641, Peso = 1.2910209894180298, Bias = 0.5050956606864929
Epoch 40: Loss = 0.2955149710178375, Peso = 1.3519452810287476, Bias = 0.5462570786476135
Epoch 50: Loss = 0.24965819716453552, Peso = 1.406904697418213, Bias = 0.5850027799606323
Epoch 60: Loss = 0.21125559508800507, Peso = 1.4566148519515991, Bias = 0.6212797164916992
Epoch 70: Loss = 0.1790657341480255, Peso = 1.5016776323318481, Bias = 0.6551027894020081
Epoch 80: Loss = 0.15206684172153473, Peso = 1.542603850364685, Bias = 0.6865332722663879
Epoch 90: Loss = 0.12941238284111023, Peso = 1.5798311233520508, Bias = 0.715662956237793
Peso final: tensor([[1.6105]], requires_grad=True)
Bias final: tensor([0.7400], requires_grad=True)
